# Fine-tuning a Language Model with LoRA

## Resources

In [1]:
# Uninstall tensorflow to avoid conflicts with PyTorch
!pip uninstall tensorflow tensorflow-cpu -y

Found existing installation: tensorflow 2.20.0
Uninstalling tensorflow-2.20.0:
  Successfully uninstalled tensorflow-2.20.0


In [3]:
# Explicitly tell transformers not to look for TensorFlow or JAX
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_JAX"] = "1"


In [38]:
import sys
# Aggressively remove tensorflow from sys.modules if it was previously loaded
# This helps prevent issues with residual or partial TensorFlow installations.
if "tensorflow" in sys.modules:
    print("WARNING: 'tensorflow' found in sys.modules. Removing it to prevent conflicts.")
    del sys.modules["tensorflow"]
for key in list(sys.modules.keys()):
    if key.startswith("tensorflow."):
        print(f"WARNING: 'tensorflow.{key}' found in sys.modules. Removing it.")
        del sys.modules[key]

# Install necessary libraries with compatible versions
%pip install --upgrade peft
%pip install datasets
%pip install transformers==4.41.0
%pip install accelerate

!mkdir -p cache

  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
INFO: pip is looking at multiple versions of tokenizers to determine which version is compatible with other requirements. This could take a while.
  Using cached transformers-5.12.1-py3-none-any.whl.metadata (33 kB)
Using cached transformers-5.12.1-py3-none-any.whl (11.2 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 36.3 MB/s eta 0:00:00
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.14.1
    Uninstalling tokenizers-0.14.1:
      Successfully uninstalled tokenizers-0.14.1
  Attempting uninstall: transformers
    Found existing installation: transformers 4.35.0
    Uninstalling transformers-4.35.0:
      Successfully uninstalled transformers-4.35.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.8/43.8 kB 2.3 MB/s eta 0:00:00
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 21.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 28.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.21.0
    Uninstalling huggingface_hub-1.21.0:
      Successfully uninstalled huggingface_hub-1.21.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.12.1
    Uninstalling transformers-5.12.1:
      Successfully uninstalled transformers-5.12.1


## Load Model and Tokenizer

In [5]:
from datasets import load_dataset
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# --- Patch for TensorFlow conflict (after transformers import) ---
# This ensures transformers does not try to use TensorFlow's random module
# even if a partial/broken TensorFlow installation is detected.
import transformers # Import here after environment variables and clean-up
try:
    from transformers.utils import is_tf_available as original_is_tf_available
    if original_is_tf_available():
        print("WARNING: TensorFlow detected by transformers. Patching is_tf_available to False.")
        transformers.utils.is_tf_available = lambda: False
        # For robustness, check and patch in trainer_utils as well if it has its own
        try:
            from transformers.trainer_utils import is_tf_available as trainer_original_is_tf_available
            if trainer_original_is_tf_available():
                transformers.trainer_utils.is_tf_available = lambda: False
        except ImportError:
            pass
except ImportError:
    print("transformers.utils.is_tf_available not found, skipping patch.")
# --- End of Patch ---

model_name = 'bigscience/bloomz-560m'
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Set pad_token to eos_token if it doesn't exist
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    if tokenizer.eos_token is None:
        tokenizer.add_special_tokens({'pad_token': '[PAD]'}) # Add a pad token if none exists

foundation_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto" if torch.cuda.is_available() else None, # Use GPU if available
    torch_dtype="auto", # Use appropriate dtype for performance
    cache_dir='./cache' # Cache directory for models
)

# Resize token embeddings if a new pad token was added
if tokenizer.pad_token_id is not None and len(tokenizer) > foundation_model.config.vocab_size:
    foundation_model.resize_token_embeddings(len(tokenizer))

print("Model and tokenizer loaded successfully.")

transformers.utils.is_tf_available not found, skipping patch.


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/222 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/14.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/85.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/715 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

Model and tokenizer loaded successfully.


## Load and Preprocess Dataset

In [6]:
data = load_dataset("Abirate/english_quotes", split="train") # Sample 10%
data = data.shuffle(seed=42).select(range(int(len(data) * 0.1)))

def tokenize_function(samples):
    return tokenizer(samples["quote"], truncation=True, padding="max_length", max_length=64)

data = data.map(tokenize_function, batched=True)

# Remove columns that are not in input_ids or attention_mask
columns_to_remove = [col for col in data.column_names if col not in ["input_ids", "attention_mask"]]
data = data.remove_columns(columns_to_remove)

train_sample = data
display(train_sample.to_pandas().head())

README.md:   0%|          | 0.00/5.55k [00:00<?, ?B/s]

quotes.jsonl:   0%|          | 0.00/647k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2508 [00:00<?, ? examples/s]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

,input_ids,attention_mask
0,"[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
1,"[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
2,"[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
3,"[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."
4,"[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, ...","[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ..."


## Configure and Apply LoRA

In [7]:
from peft import LoraConfig, get_peft_model

# Define target modules based on the model (for bloomz-560m, it's 'query_key_value')
lora_target_modules = ["query_key_value"]

lora_config = LoraConfig(
    r=8,
    lora_alpha=16, # a scaling factor that adjusts the magnitude of the weight matrix. Usually set to 1
    target_modules=lora_target_modules,
    lora_dropout=0.05,
    bias="none", # this specifies if the bias parameter should be trained.
    task_type="CAUSAL_LM"
)

#Add the adapter layers to the foundation model to be trained
peft_model = get_peft_model(foundation_model, lora_config)
print(peft_model.print_trainable_parameters())

trainable params: 786,432 || all params: 560,001,024 || trainable%: 0.14043402892063284
None


## Set Up Training Arguments and Train the Model

In [40]:
import os

output_directory = os.path.join("cache/working", "peft_lab_outputs")
os.makedirs(output_directory, exist_ok=True)
print(f"Output directory set to: {output_directory}")

Output directory set to: cache/working/peft_lab_outputs
